# Phase 1 — Traditional Computer Vision Star Detection Demonstration

## Section 1 — Project Introduction

This notebook demonstrates the completed **Phase 1 traditional computer-vision baseline** for the capstone project:
**Vision-Based Constellation Detection and Recognition Using Hybrid Computer Vision and Deep Learning**.

### Phase 1 Objective
Establish a reproducible, modular, classical computer-vision baseline for **star candidate detection** from night-sky photographs before introducing PyTorch deep-learning models in Phase 2.

### Evaluated Detectors
All three detectors operate on the same working-resolution image (`max_dim = 800`) and apply shared `cKDTree` Non-Maximum Suppression (`min_dist_px = 10.0`, `max_candidates = 250`):

1. **Global Threshold (`global_threshold`)**: Grayscale conversion, 5×5 Gaussian blur, clipped global statistical threshold `clip(mean + 2.5 * std, 55, 180)`, 2×2 morphological opening, external contour filtering (`area <= 350`), moment-based centroids, and brightness-area scoring.
2. **Adaptive Top-Hat (`CV2_AdaptiveTopHat`)**: 19×19 elliptical white Top-Hat background suppression, 3×3 (`sigma = 0.8`) blur, `[0, 255]` normalization, Gaussian adaptive thresholding combined with a global noise-floor threshold (`max(28.0, median + 3.2 * std)`), 2×2 opening, 8-connected components (`2 <= area <= 180`), and flux-weighted centroids.
3. **Multi-Scale Difference of Gaussians (`CV3_MultiScaleDoG`)**: 19×19 elliptical white Top-Hat preprocessing followed by `skimage.feature.blob_dog` multi-scale blob detection (`min_sigma = 1.1`, `max_sigma = 4.0`, `sigma_ratio = 1.5`, `threshold = 0.16`), local patch flux-weighted centroids, and scale-weighted scoring.

> **Scope Clarification**: Phase 1 detects **candidate stars only**—it does **not** perform constellation recognition, geometric pattern matching, or deep learning. Exported CSV files contain unverified detector predictions, not ground-truth annotations.

## Section 2 — Dataset & Environment Configuration

Paths are resolved via `constellation_detection.config.ProjectPaths` (configurable via `CONSTELLATION_DATA_DIR` and `CONSTELLATION_OUTPUT_DIR`) rather than hardcoded local or Colab paths.

### Reference ESA/Hubble Astronomical Sample Images
- `cassiopeia_opo0613c.jpg` — https://cdn.esahubble.org/archives/images/large/opo0613c.jpg
- `cygnus_heic0720c.jpg` — https://cdn.esahubble.org/archives/images/large/heic0720c.jpg
- `orion_opo0205b.jpg` — https://cdn.esahubble.org/archives/images/large/opo0205b.jpg
- `ursa_major_opo0706b.jpg` — https://cdn.esahubble.org/archives/images/large/opo0706b.jpg
- `ursa_minor_heic0401g.jpg` — https://cdn.esahubble.org/archives/images/large/heic0401g.jpg

If any images are not present in `data/raw/`, download them explicitly from the repository root with:
```bash
python scripts/download_phase1_samples.py
```
This notebook does **not** download external files automatically.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure

try:
    from IPython.display import display
except ImportError:
    display = print

from constellation_detection.config import ProjectPaths
from constellation_detection.evaluation import (
    build_detector_leaderboard,
    build_image_output_stem,
    evaluate_image_detectors,
    resize_to_max_dim,
)
from constellation_detection.image import load_image
from constellation_detection.visualization import (
    DETECTOR_COLORS,
    plot_detector_diagnostics,
)

# Configurable project paths
paths = ProjectPaths.from_paths()
RAW_DATA_DIR = paths.data_dir / "raw"
PREDICTIONS_DIR = paths.output_dir / "predictions"
VISUALIZATIONS_DIR = paths.output_dir / "visualizations"

# Shared Phase 1 evaluation parameters
MAX_DIM = 800
MIN_DIST_PX = 10.0
MAX_CANDIDATES = 250
TOP_K_SNR = 30
SYNTHETIC_SEED = 42

EXPECTED_SAMPLE_FILES = [
    "cassiopeia_opo0613c.jpg",
    "cygnus_heic0720c.jpg",
    "orion_opo0205b.jpg",
    "ursa_major_opo0706b.jpg",
    "ursa_minor_heic0401g.jpg",
]

loaded_images: list[tuple[str, np.ndarray]] = []
missing_files: list[str] = []

for filename in EXPECTED_SAMPLE_FILES:
    candidate_path = RAW_DATA_DIR / filename
    if candidate_path.exists():
        loaded_images.append((filename, load_image(candidate_path)))
    else:
        missing_files.append(filename)

if missing_files:
    print(
        f"Missing {len(missing_files)} sample image(s) in {RAW_DATA_DIR}: {missing_files}\n"
        "Run 'python scripts/download_phase1_samples.py' to download the ESA/Hubble samples."
    )

if not loaded_images:
    print("Using a deterministic seeded synthetic starfield fallback (not real astronomical data).")
    rng = np.random.default_rng(SYNTHETIC_SEED)
    h, w = 512, 640
    _, xx = np.mgrid[0:h, 0:w]
    bg = (12.0 + 18.0 * (xx / w) + rng.normal(0.0, 2.0, size=(h, w))).clip(0, 255).astype(np.uint8)
    for _ in range(35):
        cx = int(rng.integers(20, w - 20))
        cy = int(rng.integers(20, h - 20))
        radius = int(rng.integers(2, 4))
        intensity = int(rng.integers(180, 256))
        cv2.circle(bg, (cx, cy), radius, intensity, -1)
    loaded_images.append(("synthetic_starfield_demo.png", cv2.cvtColor(bg, cv2.COLOR_GRAY2BGR)))

dataset_rows = []
for name, img in loaded_images:
    orig_h, orig_w = img.shape[:2]
    work_h, work_w = resize_to_max_dim(img, max_dim=MAX_DIM).shape[:2]
    dataset_rows.append(
        {
            "image_file": name,
            "original_resolution": f"{orig_w}x{orig_h}",
            "working_resolution": f"{work_w}x{work_h}",
        }
    )

dataset_summary = pd.DataFrame(dataset_rows)
print(f"Loaded {len(loaded_images)} image(s) for Phase 1 demonstration:")
display(dataset_summary)

## Section 3 — Detection Demonstration

Each image is evaluated with all three traditional detectors at the same working resolution (`max_dim = 800`).

- **Four-panel diagnostic figures** (Input RGB + sky statistics, Preprocessed response, Detection mask, and Ranked star overlay) are saved to `outputs/visualizations/` for every `(image, detector)` pair.
- **Prediction CSV files** (`rank, x_px, y_px, flux_score`) are saved to `outputs/predictions/`.
- To keep the notebook readable, the four-panel comparison for the primary showcase image is displayed inline below (all remaining images are saved to disk).

In [ ]:
all_metrics: list[pd.DataFrame] = []
saved_png_paths: list[Path] = []

# Display inline panels for the first loaded image to avoid redundant figure clutter
SHOWCASE_IMAGE = loaded_images[0][0]

for image_name, image_bgr in loaded_images:
    metrics_df, results, working_img = evaluate_image_detectors(
        image=image_bgr,
        image_name=image_name,
        predictions_dir=PREDICTIONS_DIR,
        max_dim=MAX_DIM,
        min_dist_px=MIN_DIST_PX,
        max_candidates=MAX_CANDIDATES,
        top_k_snr=TOP_K_SNR,
    )
    all_metrics.append(metrics_df)

    image_id = build_image_output_stem(image_name)
    for res in results:
        detector_name = res["detector"]
        fig, saved_png = plot_detector_diagnostics(
            image_bgr=working_img,
            result=res,
            output_dir=VISUALIZATIONS_DIR,
            filename=f"{image_id}_{detector_name}_diagnostics.png",
            max_dim=MAX_DIM,
        )
        if saved_png is not None:
            saved_png_paths.append(saved_png)
        if image_name == SHOWCASE_IMAGE:
            display(fig)

print(
    f"Exported {len(saved_png_paths)} diagnostic PNGs to {VISUALIZATIONS_DIR} "
    f"and {sum(len(df) for df in all_metrics)} prediction CSVs to {PREDICTIONS_DIR}."
)

## Section 4 — Quantitative Comparison

The tables and summary bar chart below compare the three detectors across all loaded images using ground-truth-free diagnostics:

- **`total_stars`**: Number of star candidates retained after NMS (capped at `max_candidates = 250`). **Not detection accuracy and not the true star count**—higher counts may include faint noise or nebulosity artifacts, and `CV2_AdaptiveTopHat` reaches the `250`-detection cap on all five sample images.
- **`top30_snr`**: Mean local annulus Signal-to-Noise Ratio across the top 30 interior detections. **A signal-quality diagnostic, not precision or recall.**
- **`latency_ms`**: Single-image CPU runtime in milliseconds (computational cost).

> **Ground-Truth Limitation**: The ESA/Hubble sample images do not include independently verified star-level annotations. Precision, recall, F1, and localization accuracy are intentionally omitted until verified annotations are introduced.

In [ ]:
combined_metrics = pd.concat(all_metrics, ignore_index=True)
leaderboard = build_detector_leaderboard(combined_metrics)

display_cols = ["image_file", "resolution", "detector", "total_stars", "top30_snr", "latency_ms"]
print("=== Per-Image Detector Metrics ===")
display(combined_metrics[display_cols])

print("\n=== Aggregated Detector Comparison (Mean Across Images) ===")
display(leaderboard)

# Render concise 3-panel summary bar chart (values match leaderboard exactly)
summary_fig = Figure(figsize=(15.5, 5.2), dpi=120)
FigureCanvasAgg(summary_fig)
axes = summary_fig.subplots(1, 3)

detectors = leaderboard["detector"].tolist()
colors = [DETECTOR_COLORS.get(d, "steelblue") for d in detectors]
chart_specs = [
    ("total_stars", "Mean Detected Stars", "Candidate Count", "{:.1f}"),
    ("top30_snr", "Mean Top-30 Annulus SNR", "Signal-to-Noise Ratio", "{:.2f}"),
    ("latency_ms", "Mean Runtime (ms)", "Latency (ms)", "{:.2f}"),
]

# Track which detectors hit the configured NMS cap on every evaluated image
capped_all_images = {
    det: bool((combined_metrics.loc[combined_metrics["detector"] == det, "total_stars"] >= MAX_CANDIDATES).all())
    for det in detectors
}

for ax, (col, title, ylabel, fmt) in zip(axes, chart_specs, strict=True):
    values = leaderboard[col].to_numpy()
    bars = ax.bar(detectors, values, color=colors, edgecolor="black", linewidth=0.8)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=12)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    y_max = float(np.max(values)) if len(values) > 0 else 1.0
    if col == "total_stars":
        ax.axhline(
            MAX_CANDIDATES,
            color="crimson",
            linestyle="--",
            linewidth=1.2,
            label=f"NMS cap ({MAX_CANDIDATES})",
        )
        ax.set_ylim(0, max(y_max, float(MAX_CANDIDATES)) * 1.18)
        ax.legend(loc="lower right", fontsize=8, framealpha=0.9)
    else:
        ax.set_ylim(0, y_max * 1.16 if y_max > 0 else 1.0)

    for det, bar, val in zip(detectors, bars, values, strict=True):
        label = fmt.format(val)
        if col == "total_stars" and capped_all_images.get(det, False):
            label = f"{label} (cap)"
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + (ax.get_ylim()[1] * 0.02),
            label,
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold" if "(cap)" in label else "normal",
        )

summary_fig.suptitle(
    f"Phase 1 Traditional Detector Comparison (Mean Across {len(loaded_images)} Images, max_dim={MAX_DIM})",
    fontsize=12,
    fontweight="bold",
)
summary_fig.text(
    0.5,
    0.01,
    f"Note: CV2_AdaptiveTopHat reaches the configured {MAX_CANDIDATES}-detection NMS cap on all {len(loaded_images)} images "
    "(capped candidate count is not the true number of stars).",
    ha="center",
    fontsize=9,
    style="italic",
)
summary_fig.tight_layout(rect=(0.0, 0.05, 1.0, 0.93))
VISUALIZATIONS_DIR.mkdir(parents=True, exist_ok=True)
summary_png = VISUALIZATIONS_DIR / "phase1_detector_comparison_summary.png"
summary_fig.savefig(summary_png, bbox_inches="tight")
display(summary_fig)
print(f"Saved summary chart: {summary_png}")

## Section 5 — Findings & Practical Trade-Offs

Across the five ESA/Hubble reference images, each classical detector exhibits distinct trade-offs rather than one detector being universally best:

- **Detection Quantity (`total_stars`)**:
  - `CV2_AdaptiveTopHat` hits the configured `250`-detection NMS cap on **all five** ESA/Hubble sample images (mean `250.0`). This reflects the upper candidate limit rather than the true number of stars in those fields, as local background subtraction + adaptive thresholding surfaces many faint candidates across uneven nebulosity.
  - `global_threshold` (hits the cap on 3 of 5 images) and `CV3_MultiScaleDoG` (hits the cap on 2 of 5 images) return fewer candidates on average across the five-image set.
- **Signal-to-Noise Diagnostic (`top30_snr`)**:
  - `CV3_MultiScaleDoG` typically achieves the highest mean Top-30 annulus SNR, followed closely by `CV2_AdaptiveTopHat`, while `global_threshold` shows slightly lower SNR when bright extended nebulosity raises the global threshold or introduces non-stellar contours.
- **Computational Cost (`latency_ms`)**:
  - `global_threshold` is the fastest baseline, making it suitable for rapid CPU screening.
  - `CV2_AdaptiveTopHat` adds moderate morphological and connected-component overhead.
  - `CV3_MultiScaleDoG` is the slowest by a wide margin due to multi-scale Gaussian pyramid evaluation in `blob_dog`.

*Interpretation caveat*: Without verified star catalogs for these images, these diagnostics characterize **relative behavior and runtime cost**, not absolute precision or recall.

## Section 6 — Conclusion & Next Phase

### What Phase 1 Successfully Implemented
- A clean, modular, tested Python package (`constellation_detection`) covering OpenCV image I/O, preprocessing, three traditional star detectors, `cKDTree` NMS, annulus SNR diagnostics, multi-detector benchmarking, collision-free prediction CSV exports, and headless four-panel diagnostic visualizations.
- Full local CPU execution and reproducibility with 133 automated unit tests.

### Remaining Limitations
- Classical detectors rely on fixed morphological/intensity heuristics and can be affected by bright nebulosity, lens gradients, or dense star clusters.
- Sample images lack verified ground-truth star labels, preventing precision/recall/F1 evaluation.
- Constellation identification and geometric pattern matching are not yet implemented.

### Value as a Baseline & Planned Phase 2
- Phase 1 provides an immutable, well-tested reference baseline for future comparisons.
- **Phase 2** will separately investigate **PyTorch-based deep-learning star detection** (trained on Colab GPUs with local CPU inference compatibility), benchmarked directly against this Phase 1 baseline.